<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/04-evals/02-llm-as-judge.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM 作为评判模型

**目标：** 编写 judge prompt，度量它与人工标注的一致率，并找出 judge 的失败模式。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（就是本仓库）带有共享的环境准备助手，并会拉入 `groq` 客户端；本 notebook 还会用到 `sentence-transformers`、`numpy`。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领一个免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的 **钥匙图标** → **Add new secret** 添加它，名称必须严格写成 `GROQ_API_KEY`，粘贴值，然后打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整走查和选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" sentence-transformers numpy

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），并返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

> **🔵 撞上速率限制？换模型。** 这个 notebook 很费 token（每个 case 一次生成调用*加*一次 judge 调用，跑完整个集合），而 Groq 免费档的每分钟 token 上限和每日 token 上限都是**按模型**计算的。如果你收到 `429`，换一个预算全新的模型。这里有*两个*旋钮：
> ```python
> client, MODEL = setup(model='openai/gpt-oss-20b')   # 被测系统（生成）
> JUDGE_MODEL = 'qwen/qwen3.6-27b'                     # 在下面的 judge 单元格里设置
> ```
> 让 judge 和被测系统用**不同**的模型（见下面的自我偏好那一课）。每个免费模型都有自己的限制，所以从 [Groq 速率限制页](https://console.groq.com/docs/rate-limits) 挑还有余量的。只为验证跑通的话，任何有能力的对话模型都行；避开 agentic 的 `groq/compound*` 模型，它们会浏览网页并运行代码。

## 为什么需要一个 judge

上一个 notebook 里的确定性检查只停在表面：关键词在不在、文档检索对不对。它们没法告诉你一个答案是否*忠实*（没有超出素材支撑范围的论断），或者是否*完整*（覆盖了参考答案的每个要点）。参考答案没法做字符串匹配，因为一个正确的答案可以有一百种说法。

所以我们用一个模型来给答案打分。这行得通，但 judge 本身也是一个 LLM，有它自己的失败模式，而大多数团队都跳过了让它可信的那一步：用人工判断来校准它。我们会先构建 judge，然后故意把它弄坏，再校准它。


## 被测系统

接下来两个单元格以紧凑形式重建第 03 节的 RAG 系统：下载并清洗十份 IETF RFC，按章节分块，用一个小型本地模型做 embedding，按余弦相似度检索，并带引用作答。

这是一次刻意的复制，而不是 import。每个 notebook 都必须能在全新的 Colab 运行时里从上到下跑通，对教学材料来说，自包含比 DRY 更重要。如果这些细节你不熟，先过一遍 `03-rag/`；这里它们只是我们要评测的对象。


In [ ]:
# 03-rag pipeline 的紧凑版复制。自包含优先于 DRY：本 notebook 必须能
# 在 Colab 里独立跑通。这里每个设计决策的完整走查见 03-rag/。
import os
import re
import urllib.request

RFCS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
os.makedirs('data/rfc', exist_ok=True)
for n in RFCS:
    path = f'data/rfc/rfc{n}.txt'
    if not os.path.exists(path):  # 如果前面的 notebook 已经缓存过就跳过
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)

def clean_rfc(text):
    """去掉换页符和 RFC txt 文件自带的页眉/页脚行。"""
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.search(r'\[Page \d+\]\s*$', l)          # 页脚
             and not re.match(r'^\s*RFC \d+.*\d{4}\s*$', l)]   # 页眉
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines))

def chunk_rfc(text, rfc, max_chars=2000):
    """按章节分块：按带编号的标题切分，然后限制块的大小。"""
    parts = re.split(r'\n(?=\d+(?:\.\d+)*\.?\s+[A-Z])', text)
    chunks = []
    for part in parts:
        part = part.strip()
        while len(part) > max_chars:
            cut = part.rfind('\n\n', 0, max_chars)
            cut = cut if cut > 200 else max_chars
            chunks.append({'rfc': rfc, 'text': part[:cut].strip()})
            part = part[cut:].strip()
        if len(part) > 100:
            chunks.append({'rfc': rfc, 'text': part})
    return chunks

chunks = []
for n in RFCS:
    with open(f'data/rfc/rfc{n}.txt') as f:
        chunks += chunk_rfc(clean_rfc(f.read()), n)
print(f'{len(chunks)} chunks from {len(RFCS)} RFCs')

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
emb = embedder.encode([c['text'] for c in chunks],
                      normalize_embeddings=True, show_progress_bar=True)

def retrieve(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = emb @ q  # 余弦相似度：向量已做单位归一化
    return [chunks[i] for i in np.argsort(-scores)[:k]]

def answer_question(question, k=5):
    """检索 top-k 个块，带引用作答。返回 (answer, hits)。"""
    hits = retrieve(question, k)
    context = '\n\n'.join(f"[RFC {h['rfc']}]\n{h['text']}" for h in hits)
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[
            {'role': 'system',
             'content': ('Answer using ONLY the provided RFC excerpts. Cite the RFC number for '
                         'each claim, e.g. (RFC 9110). If the excerpts do not contain the answer, '
                         'say plainly that the corpus does not cover it. Do not guess.')},
            {'role': 'user', 'content': f'{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content, hits

和 notebook 01 相同的黄金集（自包含复制；每个 case 背后的理由见那个 notebook）。


In [ ]:
# 黄金集：针对真实语料写的 ~15 个 case。每个 case 都固定了
# 正确答案必须包含什么（must_mention），以及检索器应当命中
# 哪份 RFC（source_rfc）。无法回答的 case 用 source_rfc=None —— 正确
# 行为是拒答，而幻觉出一个答案是硬失败。
GOLDEN_SET = [
    {'question': 'How does the TCP three-way handshake establish a connection?',
     'reference_answer': 'The initiator sends a SYN carrying its initial sequence number; '
                         'the peer responds with SYN-ACK carrying its own sequence number and '
                         'acknowledging the first; the initiator replies with an ACK. Both '
                         'sides now have synchronized sequence numbers.',
     'source_rfc': 793, 'must_mention': ['SYN', 'ACK', 'sequence'], 'unanswerable': False},
    {'question': 'What does the HTTP 404 status code mean?',
     'reference_answer': '404 Not Found: the origin server did not find a current '
                         'representation for the target resource, or is not willing to '
                         'disclose that one exists.',
     'source_rfc': 9110, 'must_mention': ['404', 'not found'], 'unanswerable': False},
    {'question': 'What are the four roles defined in the OAuth 2.0 framework?',
     'reference_answer': 'Resource owner, resource server, client, and authorization server.',
     'source_rfc': 6749,
     'must_mention': ['resource owner', 'client', 'authorization server', 'resource server'],
     'unanswerable': False},
    {'question': 'What is the difference between an A record and a CNAME record in DNS?',
     'reference_answer': 'An A record maps a name to a 32-bit IPv4 host address; a CNAME '
                         'record maps an alias to the canonical name of its target.',
     'source_rfc': 1035, 'must_mention': ['address', 'canonical'], 'unanswerable': False},
    {'question': 'How does a client upgrade an HTTP connection to a WebSocket connection?',
     'reference_answer': 'The client sends a GET request with Upgrade: websocket and '
                         'Connection: Upgrade headers plus a Sec-WebSocket-Key; the server '
                         'replies 101 Switching Protocols with a Sec-WebSocket-Accept value '
                         'derived from the key.',
     'source_rfc': 6455, 'must_mention': ['Upgrade', '101', 'Sec-WebSocket-Key'],
     'unanswerable': False},
    {'question': 'Which SMTP commands send a mail message, and in what order?',
     'reference_answer': 'MAIL FROM identifies the sender, one or more RCPT TO commands name '
                         'recipients, then DATA introduces the message text, terminated by a '
                         'line containing only a period.',
     'source_rfc': 5321, 'must_mention': ['MAIL', 'RCPT', 'DATA'], 'unanswerable': False},
    {'question': 'What is the purpose of the Time to Live field in the IPv4 header?',
     'reference_answer': 'TTL is an upper bound on datagram lifetime. Every module that '
                         'processes the datagram decrements it, and the datagram is discarded '
                         'when it reaches zero, so packets cannot loop forever.',
     'source_rfc': 791, 'must_mention': ['zero', 'discard'], 'unanswerable': False},
    {'question': 'What is the AS_PATH attribute in BGP and why does it matter?',
     'reference_answer': 'AS_PATH records the sequence of autonomous systems a route '
                         'advertisement has traversed. BGP uses it for loop detection (a '
                         'router rejects routes containing its own AS number) and in route '
                         'selection.',
     'source_rfc': 4271, 'must_mention': ['autonomous system', 'loop'], 'unanswerable': False},
    {'question': 'How does HTTP/2 carry multiple requests over a single connection?',
     'reference_answer': 'HTTP/2 multiplexes exchanges as independent streams over one '
                         'connection. Each frame carries a stream identifier, so concurrent '
                         'requests and responses interleave without blocking each other.',
     'source_rfc': 7540, 'must_mention': ['stream', 'frame'], 'unanswerable': False},
    {'question': 'Which HTTP methods are defined as idempotent?',
     'reference_answer': 'PUT and DELETE, plus all safe methods: GET, HEAD, OPTIONS, TRACE.',
     'source_rfc': 9110, 'must_mention': ['PUT', 'DELETE', 'GET'], 'unanswerable': False},
    {'question': 'Why does a TCP connection enter the TIME-WAIT state, and for how long?',
     'reference_answer': 'The side that closes actively waits in TIME-WAIT for twice the '
                         'maximum segment lifetime (2*MSL), so delayed segments from the old '
                         'incarnation die off before the socket pair can be reused.',
     'source_rfc': 793, 'must_mention': ['TIME-WAIT', 'MSL'], 'unanswerable': False},
    {'question': 'Walk through the OAuth 2.0 authorization code grant flow.',
     'reference_answer': 'The client redirects the resource owner to the authorization '
                         'server; after authentication and consent the server redirects back '
                         'with an authorization code; the client exchanges the code, with its '
                         'own credentials, at the token endpoint for an access token.',
     'source_rfc': 6749, 'must_mention': ['authorization code', 'access token', 'redirect'],
     'unanswerable': False},
    {'question': 'What does HTTP status 503 indicate, and which header can accompany it?',
     'reference_answer': '503 Service Unavailable: the server is currently unable to handle '
                         'the request, due to overload or maintenance. It may send Retry-After '
                         'to indicate how long to wait.',
     'source_rfc': 9110, 'must_mention': ['503', 'Retry-After'], 'unanswerable': False},
    # --- 刻意设为无法回答：正确行为是拒答 ---
    {'question': 'How does QUIC combine the transport and cryptographic handshakes?',
     'reference_answer': 'Declines: QUIC (RFC 9000) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'What cipher suites does TLS 1.3 define?',
     'reference_answer': 'Declines: TLS 1.3 (RFC 8446) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'Which port does IMAP use, and how does a client select a mailbox?',
     'reference_answer': 'Declines: IMAP is not in the corpus (SMTP is the only mail '
                         'protocol here).',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
]
print(len(GOLDEN_SET), 'cases,',
      sum(c['unanswerable'] for c in GOLDEN_SET), 'unanswerable')


## 构建 judge

judge 就是一个 LLM，所以它有 LLM 的失败模式。咬人最狠的三种，以及各自对应的设计决策，是一一对应的。我们先把三种缓解措施全都装进 judge，然后（在后面）逐个拆掉，让你亲眼看到失败出现：

| judge 失败模式 | 它长什么样 | 对应的设计决策 |
|---|---|---|
| **冗长偏好** | 冗长、灌水的答案得分高于简洁但正确的答案 | rubric 规定「评判实质，不评判风格；多出来的长度不加分」 |
| **自我偏好** | 模型给自己的*输出*打更高分 | 用和被测系统*不同*的模型来评判 |
| **分数压缩** | 一切都挤在 3–4 分；回归就此消失 | 把 1–5 的每个分数锚定到明确定义 |

外加两个让分数真正可用的结构性选择：

- **通过强制工具调用来做结构化输出。** judge 填一个 schema（`{reasoning, faithful, complete, score}`）而不是自由文本，这样测试骨架无需解析散文就能聚合。`tool_choice` 强制发起这次调用，所以我们总能拿到一个分数。
- **先推理，后给分。** schema 把 `reasoning` 放在最前面，prompt 里也明确这么要求。如果模型先定下分数，推理就变成对一个草率判断的事后合理化；先推理能可测量地提升分数质量。顺序很重要。

> **⚠️ 生产现实 ——** 强制 `tool_choice` 会让模型把这次调用当作文本吐出来，再由 API 解析；当解析失败时，Groq 返回的是 **400 `tool_use_failed`**，而不是一条消息。它是*间歇性*的（同一个输入下一次可能就成功），所以 `judge()` 把这次调用包进一个短重试，并在失败时回退到哨兵分数 0，而不是让整轮运行崩溃。这正是这个 notebook 自己讲的那一课用到自己身上：judge 是一次不可靠的 LLM 调用，所以要为它做防护。裸写 `.tool_calls[0]` 会在第一个不稳定分数上把整个循环搞挂。

In [ ]:
import json

# judge 模型：用小档，不用大档。两个原因。(1) 廉价开发、在
# 目标上评测：你在迭代评测骨架本身时会不停地重跑它，所以用
# 便宜模型；在真正的发布决策之前，用更强的模型
# 或人工复核重跑一遍评判。(2) 用与被测系统不同的模型
# 可以避开自我偏好偏差（下面有更多说明）。
JUDGE_MODEL = 'openai/gpt-oss-20b'

JUDGE_TOOL = {
    'type': 'function',
    'function': {
        'name': 'grade_answer',
        'description': 'Record your grade for the candidate answer.',
        'parameters': {
            'type': 'object',
            'properties': {
                'reasoning': {'type': 'string',
                              'description': 'Compare candidate to reference point by point, '
                                             'BEFORE deciding any grade. 2-4 sentences.'},
                'faithful': {'type': 'boolean',
                             'description': 'True if every claim in the candidate is supported '
                                            'by the reference (nothing fabricated).'},
                'complete': {'type': 'boolean',
                             'description': 'True if the candidate covers every key point of '
                                            'the reference.'},
                'score': {'type': 'integer',
                          'description': 'Overall 1-5, anchored to the rubric.'},
            },
            'required': ['reasoning', 'faithful', 'complete', 'score'],
        },
    },
}

JUDGE_SYSTEM = """You grade answers from a retrieval-augmented QA system against a reference answer.

Rubric -- anchor your score to these definitions and use the full range:
1 = wrong or fabricated: contradicts the reference or invents facts
2 = mostly wrong: a relevant fragment, but the main point is missing or incorrect
3 = partially correct: main point present, but a real gap or a minor unsupported claim
4 = correct: matches the reference on every key point; only minor omissions
5 = correct and complete: nothing missing, nothing unsupported

If the reference says the correct behavior is to decline, a candidate that declines
scores 5 and a candidate that invents an answer scores 1.

Write your reasoning FIRST, then decide the score. Judge substance, not style:
a short correct answer outranks a long vague one. Extra length earns nothing."""

from groq import BadRequestError

def _grade_call(system, question, reference, candidate, tries=3):
    """一次强制的 tool judge 调用，在 Groq 的 tool_use_failed 上重试。
    强制 tool_choice 会让模型把这次调用当作文本吐出来，再由 Groq
    解析；当解析失败时它抛出一个 400（tool_use_failed），而不是
    返回一条消息。它是间歇性的，所以重试两次就能清掉。
    这正是本 notebook 教的那一课，用在 judge 自己身上：
    一次 LLM 调用不可靠，所以把它包起来。返回解析后的 dict，如果
    每次尝试都失败则返回 None。
    """
    for _ in range(tries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, max_tokens=700,
                tools=[JUDGE_TOOL],
                tool_choice={'type': 'function', 'function': {'name': 'grade_answer'}},
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content':
                     f'Question: {question}\n\nReference answer:\n{reference}'
                     f'\n\nCandidate answer:\n{candidate}'},
                ],
            )
            return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)
        except BadRequestError as e:
            if 'tool_use_failed' in str(e):
                continue          # 模型没有发出有效的调用；再试一次
            raise                 # 一个真正的 400（schema 有问题等）—— 不要掩盖它
    return None                   # 重试次数耗尽；由调用方决定怎么办

def judge(question, reference, candidate):
    """给一个答案打分。返回 {reasoning, faithful, complete, score}。"""
    grade = _grade_call(JUDGE_SYSTEM, question, reference, candidate)
    if grade is None:  # judge 重试后仍未能产出分数 —— 安全失败，不要让整轮运行崩溃
        return {'reasoning': '(judge produced no valid tool call after retries)',
                'faithful': False, 'complete': False, 'score': 0}
    return grade

## 评判一次 RAG 系统的新运行

为整个黄金集生成答案，然后逐个打分。注意 judge 拿到的是*参考答案*，不是检索到的块。它对着我们写的地面真值打分，这正是黄金集承担重任的地方。


In [ ]:
# 成本提示：每个 case 一次生成调用（120B，被测系统）+ 一次 judge 调用
# （20B，廉价的批量评判）= 2 次调用 x len(GOLDEN_SET)。
judged = []
for case in GOLDEN_SET:
    answer, hits = answer_question(case['question'])
    grade = judge(case['question'], case['reference_answer'], answer)
    judged.append({'question': case['question'], 'answer': answer,
                   'unanswerable': case['unanswerable'], 'grade': grade})
    print(f"score={grade['score']}  faithful={grade['faithful']}  "
          f"complete={grade['complete']}  | {case['question'][:55]}")

scores = [j['grade']['score'] for j in judged]
print('-' * 78)
print(f'mean score: {sum(scores) / len(scores):.2f}   '
      f'distribution: {sorted(scores)}')

跑一下，读几条 `reasoning` 字段，尤其是任何得 3 分的 case。judge 能抓住关键词覆盖率抓不到的东西：一个提到了每个关键词却把因果关系搞乱的答案，或者一个悄悄加上参考答案并不支持的论断的答案。也看看无法回答的 case，judge 处理拒答远比 notebook 01 里那个标记扫描的土办法可靠。

## 失败模式 1：冗长偏好

judge 偏好更长的答案。不加防护的话，一个灌水、含糊的答案常常得分高于简洁但正确的答案，因为长度读起来像是下了功夫。我们为 404 这个问题构造了两个答案：一个简洁且完全准确，一个又长、又热情、却没有内容（它其实从没说过 "not found"）。把两个都评判一下并对比。

In [ ]:
q = 'What does the HTTP 404 status code mean?'
ref = ('404 Not Found: the origin server did not find a current representation for '
       'the target resource, or is not willing to disclose that one exists.')

terse = ('404 Not Found: the server has no current representation for the target '
         'resource, or will not disclose that one exists.')

padded = ('Great question! HTTP status codes are a fascinating and essential part of '
          'how the web communicates outcomes. The 404 code belongs to the 4xx class, '
          'which broadly indicates that something about the client\'s request was '
          'problematic in some way. Codes in this family are extremely common in '
          'day-to-day web browsing, and 404 in particular is one nearly every user '
          'has encountered. It generally relates to the situation of the requested '
          'resource on the server side of the exchange, and understanding it is an '
          'important part of debugging web applications effectively.')

for name, cand in [('terse-but-correct', terse), ('padded-but-vague', padded)]:
    g = judge(q, ref, cand)
    print(f"{name:18} score={g['score']}  faithful={g['faithful']}  "
          f"complete={g['complete']}")
    print('  ', g['reasoning'][:160])


跑的时候注意这个差距。用锚定 rubric 的 prompt（「评判实质，不评判风格」「多出来的长度不加分」），简洁的答案应当明显胜出。如果你从 `JUDGE_SYSTEM` 里删掉那两行再重跑，灌水答案的分数通常会往上爬：这个偏差是真的，而 prompt 里的防护是在起作用。灌水答案从没说出那唯一重要的事实，可它*听起来*像一个答案，而这正是没有 prompt 防护的 judge 会奖励的东西。

## 失败模式 2：自我偏好

一个模型评判自己的输出时会打更高分，因为生成这个答案的风格先验也会替它打分。这就是为什么我们的 judge 是 8B 模型，而被测系统是 70B。注意这**推翻了通常的建议**，即 judge 至少要和它评判的系统一样强。这个反转是刻意的权衡：8B 抓不到 70B 能抓到的细微不完整，但手里有参考答案时，打分比作答容易得多，而且一个弱 judge 把候选与参考答案对比，仍能可靠抓住严重失败（幻觉、漏掉要点、该拒答却不拒答）。廉价地开发；在发布决策之前，用更强的模型重跑一遍评判，并人工复核分歧。

## 失败模式 3：分数压缩

judge 会回避标尺的两端。要 1–5 分却不给任何定义，你得到的是一个挤在 3–4 分的分布，这会毁掉指标检测回归的能力（一切都永远是 3.5）。缓解措施已经在 `JUDGE_SYSTEM` 里了：每个分数都是*有定义的*。下面是同样一次打分，用一个没有锚定的 prompt，让你直接看到压缩。


In [ ]:
# 一个没有锚定的 judge：同样的 schema，没有 rubric，没有先推理的指令。
# 复用 _grade_call，所以它和真正的 judge 一样有 tool_use_failed 重试。
def judge_unanchored(question, reference, candidate):
    grade = _grade_call('Grade the candidate answer against the reference on a 1-5 scale.',
                        question, reference, candidate)
    return grade or {'score': 0}

# 为 TIME-WAIT 问题准备的三个质量明显不同的候选答案。
q2 = 'Why does a TCP connection enter the TIME-WAIT state, and for how long?'
ref2 = ('The side that closes actively waits in TIME-WAIT for twice the maximum '
        'segment lifetime (2*MSL), so delayed segments from the old incarnation die '
        'off before the socket pair can be reused.')
candidates = {
    'good': ('The active closer waits 2*MSL in TIME-WAIT so that stray segments '
             'from the old connection expire before the port pair is reused.'),
    'partial': ('TIME-WAIT is a state a TCP connection passes through while '
                'closing; it lasts a fixed amount of time.'),
    'wrong': ('TIME-WAIT lets the receiving side finish processing buffered data; '
              'it lasts exactly 30 seconds on all implementations.'),
}

# 6 次 judge 调用（小档）。
print(f"{'candidate':10} {'anchored':>9} {'unanchored':>11}")
for name, cand in candidates.items():
    a = judge(q2, ref2, cand)['score']
    u = judge_unanchored(q2, ref2, cand)['score']
    print(f'{name:10} {a:>9} {u:>11}')

## 练习

1. **用一个自信的幻觉弄坏 judge。** 为 BGP 问题写一个候选答案：流畅、结构良好、引用 "(RFC 4271)"，但事实错误（比如声称 AS_PATH 用于加密）。用锚定和未锚定的 judge 各评判一次。有哪个会给出及格分数吗？
2. **度量位置偏差。** 给 judge 做一个成对比较的变体：一个带 `winner: 'A' | 'B'` 的工具 schema，输入两个候选答案。在简洁/灌水这一对上跑两次，一次把简洁的放 A，一次把它放 B，覆盖全部 13 个可回答问题生成的答案对参考答案。判决随顺序翻转的频率有多高？
3. **judge 模型扫描。** 把 `JUDGE_MODEL` 设为 `openai/gpt-oss-120b` 重跑这 16 个 case 的评判，并计算与 20B 那次逐个分数的一致率。它们在哪里分歧：随机 case，还是在 3 分对 4 分的边界上系统性地分歧？
4. **加一个 `citations_valid` 字段。** 给 `JUDGE_TOOL` 扩展一个布尔值，检查候选答案里引用的每个 RFC 编号是否真的出现在检索到的块里（把块列表传进 judge prompt）。这把 judge 变成一个对素材的忠实性检查器，而不只是对参考答案的检查器。

In [ ]:
# 第 1 步：读前 10 条评判输出，对每一条形成你自己的判决。
sample = judged[:10]
for i, row in enumerate(sample):
    print('=' * 78)
    print(f"[{i}] {row['question']}")
    print(f"    answer : {row['answer'][:300]}")
    print(f"    judge  : score={row['grade']['score']} "
          f"faithful={row['grade']['faithful']}")


In [ ]:
# 第 2 步：为上面每个索引记下你的判决。
#   True  = 你会接受这个答案（等价于 score >= 4）
#   False = 你不会
#   None  = 跳过
HUMAN_LABELS = {0: None, 1: None, 2: None, 3: None, 4: None,
                5: None, 6: None, 7: None, 8: None, 9: None}

labeled = {i: v for i, v in HUMAN_LABELS.items() if v is not None}
if not labeled:
    print('Fill in HUMAN_LABELS above, then rerun this cell.')
else:
    agree = sum((sample[i]['grade']['score'] >= 4) == v for i, v in labeled.items())
    print(f'agreement with judge: {agree}/{len(labeled)} = {agree / len(labeled):.0%}')
    for i, v in labeled.items():
        jv = sample[i]['grade']['score'] >= 4
        if jv != v:
            print(f'  DISAGREE [{i}] you={v} judge={jv} '
                  f'(score {sample[i]["grade"]["score"]}): '
                  f'{sample[i]["question"][:55]}')


把分歧当成真正有价值的产出，而不是一致率。每个分歧要么是一个 judge bug（修 rubric；通常是你产生分歧的那个边界上锚定定义太松），要么是一条标注洞察（你自己的标准本来就模糊，那么收紧参考答案）。低于约 80% 一致率，就不要拿这个 judge 的数字来做决策。

一个能规模化的习惯：把人工标注的样本留下来。它们会变成一个*元评测*，一个给 judge 自己的黄金集，每当 judge prompt 或模型变动时你就重跑一遍。托管的评测工具（Braintrust、LangSmith、promptfoo 之类）都有专门为这个标注循环做的界面；机制上做的就是你刚刚手工做的事。
